# 11 - Piecewise Linear Loss Functions

This notebook explores SGD dynamics on piecewise linear loss functions and their smooth approximations.

**Converted from:** `Danilo_piecewiselinlosses.nb` (Mathematica)  
**Credit:** Original analysis by Dr. Danilo Forastiere

## Contents:
1. Piecewise linear loss construction
2. Smooth approximations (softplus, sigmoid)
3. ReLU network loss landscapes
4. Comparison with smooth losses
5. Implications for SGD dynamics

## Background

Neural networks with ReLU activations create piecewise linear functions, leading to loss landscapes with flat regions and sharp transitions. Understanding these non-smooth landscapes is crucial for analyzing modern deep learning.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add utils to path
sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from visualization import plot_loss_landscape

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print("\n** Credit: Original Mathematica analysis by Dr. Danilo Forastiere **")

## 1. Piecewise Linear Loss Functions

We'll construct loss functions based on piecewise linear components, similar to ReLU networks.

The basic building block is:
$$\text{ReLU}(x) = \max(0, x)$$

And absolute value:
$$|x| = \max(x, -x)$$

In [ ]:
def relu(x):
    """ReLU activation function."""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of ReLU (subgradient at 0)."""
    return (x > 0).astype(float)

def abs_func(x):
    """Absolute value function."""
    return np.abs(x)

def abs_derivative(x):
    """Derivative of abs (subgradient at 0)."""
    result = np.sign(x)
    result[x == 0] = 0  # Subgradient choice at 0
    return result

# Visualize basic functions
x = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ReLU
axes[0].plot(x, relu(x), 'b-', linewidth=3, label='ReLU(x)')
axes[0].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('ReLU(x)', fontsize=12)
axes[0].set_title('ReLU Activation', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# Absolute value
axes[1].plot(x, abs_func(x), 'r-', linewidth=3, label='|x|')
axes[1].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('|x|', fontsize=12)
axes[1].set_title('Absolute Value', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## 2. Smooth Approximations

Piecewise linear functions are non-differentiable at breakpoints. We can use smooth approximations:

**Softplus** (smooth ReLU):
$$\text{softplus}(x) = \log(1 + e^x)$$

**Scaled sigmoid** (smooth abs):
$$\text{smooth_abs}(x) = x \cdot \tanh(\beta x) \approx |x|$$

In [ ]:
def softplus(x, beta=1.0):
    """Smooth approximation to ReLU."""
    return np.log(1 + np.exp(beta * x)) / beta

def softplus_derivative(x, beta=1.0):
    """Derivative of softplus (sigmoid)."""
    return 1.0 / (1.0 + np.exp(-beta * x))

def smooth_abs(x, beta=2.0):
    """Smooth approximation to absolute value."""
    return x * np.tanh(beta * x)

def smooth_abs_derivative(x, beta=2.0):
    """Derivative of smooth abs."""
    tanh_val = np.tanh(beta * x)
    sech2_val = 1 - tanh_val**2
    return tanh_val + beta * x * sech2_val

print("Smooth approximations defined")

In [ ]:
# Compare piecewise linear with smooth approximations
x = np.linspace(-3, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ReLU vs Softplus
axes[0].plot(x, relu(x), 'b-', linewidth=3, label='ReLU (piecewise)', alpha=0.7)
axes[0].plot(x, softplus(x, beta=1), 'r--', linewidth=2.5, label='Softplus (β=1)', alpha=0.8)
axes[0].plot(x, softplus(x, beta=5), 'g--', linewidth=2.5, label='Softplus (β=5)', alpha=0.8)
axes[0].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('f(x)', fontsize=12)
axes[0].set_title('ReLU vs Smooth Approximations', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Abs vs Smooth Abs
axes[1].plot(x, abs_func(x), 'b-', linewidth=3, label='|x| (piecewise)', alpha=0.7)
axes[1].plot(x, x * np.tanh(1*x), 'r--', linewidth=2.5, label='Smooth (β=1)', alpha=0.8)
axes[1].plot(x, x * np.tanh(5*x), 'g--', linewidth=2.5, label='Smooth (β=5)', alpha=0.8)
axes[1].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('f(x)', fontsize=12)
axes[1].set_title('Absolute Value vs Smooth Approximations', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Higher β makes smoother approximation closer to piecewise linear")

## 3. Piecewise Linear Loss Landscape

Construct a 2D loss function using piecewise linear components.

In [ ]:
class PiecewiseLinearLoss:
    """Piecewise linear loss function."""
    
    def __call__(self, params):
        """Compute piecewise linear loss."""
        a, b = params
        
        # Combination of abs and relu terms
        loss = abs_func(a - 1.0) + 0.5 * abs_func(b - 0.5)
        loss += 0.3 * relu(a + b - 1.5)
        loss += 0.2 * relu(-a + b + 0.5)
        
        return loss
    
    def gradient(self, params):
        """Compute subgradient."""
        a, b = params
        
        grad = np.zeros(2)
        
        # Subgradients
        grad[0] = abs_derivative(a - 1.0)
        grad[0] += 0.3 * relu_derivative(a + b - 1.5)
        grad[0] += 0.2 * relu_derivative(-a + b + 0.5) * (-1)
        
        grad[1] = 0.5 * abs_derivative(b - 0.5)
        grad[1] += 0.3 * relu_derivative(a + b - 1.5)
        grad[1] += 0.2 * relu_derivative(-a + b + 0.5)
        
        return grad

class SmoothPiecewiseLinearLoss:
    """Smooth approximation to piecewise linear loss."""
    
    def __init__(self, beta=5.0):
        self.beta = beta
    
    def __call__(self, params):
        """Compute smooth loss."""
        a, b = params
        
        # Smooth versions
        loss = (a - 1.0) * np.tanh(self.beta * (a - 1.0))
        loss += 0.5 * (b - 0.5) * np.tanh(self.beta * (b - 0.5))
        loss += 0.3 * softplus(a + b - 1.5, self.beta)
        loss += 0.2 * softplus(-a + b + 0.5, self.beta)
        
        return loss
    
    def gradient(self, params):
        """Compute gradient of smooth loss."""
        a, b = params
        
        grad = np.zeros(2)
        
        # Derivatives
        tanh_a = np.tanh(self.beta * (a - 1.0))
        sech2_a = 1 - tanh_a**2
        grad[0] = tanh_a + self.beta * (a - 1.0) * sech2_a
        
        tanh_b = np.tanh(self.beta * (b - 0.5))
        sech2_b = 1 - tanh_b**2
        grad[1] = 0.5 * (tanh_b + self.beta * (b - 0.5) * sech2_b)
        
        grad[0] += 0.3 * softplus_derivative(a + b - 1.5, self.beta)
        grad[0] += 0.2 * softplus_derivative(-a + b + 0.5, self.beta) * (-1)
        
        grad[1] += 0.3 * softplus_derivative(a + b - 1.5, self.beta)
        grad[1] += 0.2 * softplus_derivative(-a + b + 0.5, self.beta)
        
        return grad

loss_pw = PiecewiseLinearLoss()
loss_smooth = SmoothPiecewiseLinearLoss(beta=5.0)

print("Piecewise linear and smooth loss functions created")

In [ ]:
# Visualize both loss landscapes
param_range = ((-0.5, 2.5), (-0.5, 2.5))
n_grid = 100

a_vals = np.linspace(param_range[0][0], param_range[0][1], n_grid)
b_vals = np.linspace(param_range[1][0], param_range[1][1], n_grid)
A, B = np.meshgrid(a_vals, b_vals)

# Compute losses
L_pw = np.zeros_like(A)
L_smooth = np.zeros_like(A)

for i in range(n_grid):
    for j in range(n_grid):
        params = np.array([A[i, j], B[i, j]])
        L_pw[i, j] = loss_pw(params)
        L_smooth[i, j] = loss_smooth(params)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Piecewise linear
contourf1 = axes[0].contourf(A, B, L_pw, levels=30, cmap='viridis')
axes[0].contour(A, B, L_pw, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf1, ax=axes[0], label='Loss')
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Parameter b', fontsize=12)
axes[0].set_title('Piecewise Linear Loss Landscape', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Smooth approximation
contourf2 = axes[1].contourf(A, B, L_smooth, levels=30, cmap='viridis')
axes[1].contour(A, B, L_smooth, levels=30, colors='black', alpha=0.3, linewidths=0.5)
plt.colorbar(contourf2, ax=axes[1], label='Loss')
axes[1].set_xlabel('Parameter a', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('Smooth Approximation Loss Landscape', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice flat regions and sharp transitions in piecewise linear loss")

## 4. SGD on Piecewise Linear vs Smooth Loss

Compare SGD trajectories on both landscapes.

In [ ]:
# Run SGD on both losses
initial_params = np.array([0.2, 2.0])

def simple_sgd(loss_fn, initial_params, learning_rate, n_steps, noise_std=0.01):
    """Simple SGD with gradient noise."""
    trajectory = np.zeros((n_steps + 1, 2))
    trajectory[0] = initial_params
    
    for i in range(n_steps):
        grad = loss_fn.gradient(trajectory[i])
        noise = np.random.randn(2) * noise_std
        trajectory[i + 1] = trajectory[i] - learning_rate * (grad + noise)
    
    return trajectory

np.random.seed(42)
traj_pw = simple_sgd(loss_pw, initial_params, learning_rate=0.05, n_steps=500)

np.random.seed(42)
traj_smooth = simple_sgd(loss_smooth, initial_params, learning_rate=0.05, n_steps=500)

print("SGD trajectories computed")
print(f"Piecewise: Start {traj_pw[0]} -> End {traj_pw[-1]}")
print(f"Smooth: Start {traj_smooth[0]} -> End {traj_smooth[-1]}")

In [ ]:
# Plot trajectories on both landscapes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Piecewise linear
contourf1 = axes[0].contourf(A, B, L_pw, levels=30, cmap='viridis', alpha=0.6)
axes[0].contour(A, B, L_pw, levels=30, colors='black', alpha=0.3, linewidths=0.5)
axes[0].plot(traj_pw[:, 0], traj_pw[:, 1], 'r-', linewidth=2.5, alpha=0.8, label='SGD trajectory')
axes[0].plot(traj_pw[0, 0], traj_pw[0, 1], 'go', markersize=12, 
            markeredgecolor='black', markeredgewidth=2, label='Start')
axes[0].plot(traj_pw[-1, 0], traj_pw[-1, 1], 'rs', markersize=12,
            markeredgecolor='black', markeredgewidth=2, label='End')
plt.colorbar(contourf1, ax=axes[0], label='Loss')
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Parameter b', fontsize=12)
axes[0].set_title('SGD on Piecewise Linear Loss', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Smooth
contourf2 = axes[1].contourf(A, B, L_smooth, levels=30, cmap='viridis', alpha=0.6)
axes[1].contour(A, B, L_smooth, levels=30, colors='black', alpha=0.3, linewidths=0.5)
axes[1].plot(traj_smooth[:, 0], traj_smooth[:, 1], 'r-', linewidth=2.5, alpha=0.8, label='SGD trajectory')
axes[1].plot(traj_smooth[0, 0], traj_smooth[0, 1], 'go', markersize=12,
            markeredgecolor='black', markeredgewidth=2, label='Start')
axes[1].plot(traj_smooth[-1, 0], traj_smooth[-1, 1], 'rs', markersize=12,
            markeredgecolor='black', markeredgewidth=2, label='End')
plt.colorbar(contourf2, ax=axes[1], label='Loss')
axes[1].set_xlabel('Parameter a', fontsize=12)
axes[1].set_ylabel('Parameter b', fontsize=12)
axes[1].set_title('SGD on Smooth Approximation', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Implications for SGD Dynamics

Analyze convergence and stability on piecewise linear losses.

In [ ]:
# Compute loss evolution
losses_pw = np.array([loss_pw(params) for params in traj_pw])
losses_smooth = np.array([loss_smooth(params) for params in traj_smooth])

# Plot comparison
fig, ax = plt.subplots(figsize=(14, 7))

iterations = np.arange(len(traj_pw))

ax.plot(iterations, losses_pw, 'b-', linewidth=2.5, 
       alpha=0.8, label='Piecewise Linear Loss')
ax.plot(iterations, losses_smooth, 'r--', linewidth=2.5,
       alpha=0.8, label='Smooth Approximation Loss')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Evolution: Piecewise Linear vs Smooth', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal losses:")
print(f"  Piecewise linear: {losses_pw[-1]:.4f}")
print(f"  Smooth: {losses_smooth[-1]:.4f}")

In [ ]:
# Analyze gradient magnitudes
grad_norms_pw = np.array([np.linalg.norm(loss_pw.gradient(params)) for params in traj_pw])
grad_norms_smooth = np.array([np.linalg.norm(loss_smooth.gradient(params)) for params in traj_smooth])

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(iterations, grad_norms_pw, 'b-', linewidth=2, 
       alpha=0.7, label='Piecewise Linear')
ax.plot(iterations, grad_norms_smooth, 'r--', linewidth=2,
       alpha=0.7, label='Smooth Approximation')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Gradient Norm', fontsize=12)
ax.set_title('Gradient Magnitude Evolution', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nPiecewise linear gradients can have discontinuous jumps at boundaries")

## Summary

In this notebook, we explored piecewise linear loss functions inspired by ReLU networks:

1. **Piecewise Linear Functions**: Examined ReLU and absolute value as building blocks
2. **Smooth Approximations**: Used softplus and smooth absolute value for differentiability
3. **Loss Landscapes**: Visualized the characteristic flat regions and sharp transitions
4. **SGD Comparison**: Ran SGD on both piecewise linear and smooth versions
5. **Dynamics Analysis**: Studied convergence and gradient behavior

**Key Insights:**
- Piecewise linear losses create flat regions where gradients are constant or zero
- Sharp transitions at boundaries cause gradient discontinuities
- Smooth approximations enable standard gradient-based analysis
- ReLU networks naturally create piecewise linear loss landscapes
- SGD on piecewise linear losses can exhibit different behavior than on smooth losses

**Practical Implications:**
- ReLU networks partition parameter space into linear regions
- Gradient descent may get stuck in flat regions
- Momentum and adaptive methods help navigate non-smooth landscapes
- Smoothness of approximation (β parameter) affects optimization

**Credit:** This analysis builds on original work by Dr. Danilo Forastiere studying piecewise linear loss functions in the context of neural network optimization.

## Next Steps

This completes the series of SGD dynamics notebooks! Key topics covered:
- Basic SGD implementation and visualization
- Fluctuation-dissipation theorem and Langevin dynamics
- Escape times and barrier crossing
- Stationary distributions and Fokker-Planck equation
- Stochastic differential equations
- Parameter sweeps and statistical analysis
- Multi-dimensional examples
- Piecewise linear and non-smooth losses